In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

: 

In [ ]:
warnings.filterwarnings("ignore")

## PART : 1 - Synthetic Dataset 

In [ ]:
def generate_patient_data(n=500, seed=42):
    rng = np.random.default_rng(seed)
    risk_class = rng.choice([0, 1, 2], size=n, p=[0.50, 0.30, 0.20])

    records = []
    
    for rc in risk_class:
        if rc == 0:    # ── LOW RISK patient ──────────────────────────
            age     = int(np.clip(rng.normal(35, 8), 18, 55))    
            bmi     = round(np.clip(rng.normal(23, 2.5), 17, 29), 1)  
            glucose = round(np.clip(rng.normal(88, 8), 70, 105), 1)   
            bp      = int(np.clip(rng.normal(118, 8), 90, 135))        
            hr      = int(np.clip(rng.normal(70, 8), 55, 90))          
            sleep   = round(np.clip(rng.normal(7.2, 0.7), 6, 9), 1)   
            activity = rng.choice(["High", "Moderate"], p=[0.55, 0.45])
            smoking  = rng.choice(["Never", "Former", "Current"], p=[0.75, 0.20, 0.05])

        elif rc == 1:  # ── MEDIUM RISK patient ──────────────────────────
            age     = int(np.clip(rng.normal(48, 9), 30, 65))
            bmi     = round(np.clip(rng.normal(27.5, 3), 22, 34), 1) 
            glucose = round(np.clip(rng.normal(108, 12), 90, 135), 1) 
            bp      = int(np.clip(rng.normal(132, 10), 115, 155))
            hr      = int(np.clip(rng.normal(80, 9), 65, 100))
            sleep   = round(np.clip(rng.normal(6.2, 0.8), 4.5, 7.5), 1)
            activity = rng.choice(["Moderate", "Low"], p=[0.55, 0.45])
            smoking  = rng.choice(["Never", "Former", "Current"], p=[0.40, 0.35, 0.25])

        else:  # rc == 2  ── HIGH RISK patient ─────────────────────────
            age     = int(np.clip(rng.normal(58, 10), 35, 80))   
            bmi     = round(np.clip(rng.normal(32.5, 4), 26, 42), 1)  
            glucose = round(np.clip(rng.normal(138, 20), 110, 200), 1) 
            bp      = int(np.clip(rng.normal(148, 12), 130, 185))  
            hr      = int(np.clip(rng.normal(90, 12), 72, 130))
            sleep   = round(np.clip(rng.normal(5.2, 1.0), 3, 6.5), 1)
            activity = rng.choice(["Low", "Moderate"], p=[0.75, 0.25])
            smoking  = rng.choice(["Never", "Former", "Current"], p=[0.20, 0.30, 0.50])

        gender = rng.choice(["Male", "Female"])

        records.append({"age": age,
            "gender": gender,
            "blood_pressure": bp,
            "heart_rate": hr,
            "bmi": bmi,
            "glucose_level": glucose,
            "sleep_hours": sleep,
            "activity_level": activity,
            "smoking_status": smoking,
            "risk_label": rc  
        })

    df = pd.DataFrame(records)

    for col in ["bmi", "glucose_level", "sleep_hours"]:
        mask = rng.choice([True, False], size=n, p=[0.05, 0.95])
        df.loc[mask, col] = np.nan

    return df

print("Dataset Created...")

##### Generate the Data :

In [ ]:
df = generate_patient_data(n=500)

# Save to a CSV file (like Excel) so you can open it later
df.to_csv("synthetic_patient_dataset.csv", index=False)

print(f"Dataset created!")
print(f"Shape: {df.shape}")
print(f"{df.shape[0]} patients, {df.shape[1]} columns")
df.head()

##### Check for missing values :

In [ ]:
print(df.isnull().sum())

##### Basic statistics about our Data :

In [ ]:
df.describe()

##### How many patients in each risk group?

In [ ]:
risk_counts = df["risk_label"].value_counts().sort_index()
risk_names  = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}

In [ ]:
print("Risk distribution:")
for label, count in risk_counts.items():
    print(f"{risk_names[label]:12s} ({label}): {count:3d} patients")

## PART 2 : Exploring the Data (EDA)

In [ ]:
print(df.dtypes)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("Feature Distributions by Risk Level", fontsize=16, fontweight="bold")
risk_labels_map = {0: "Low", 1: "Medium", 2: "High"}
df["risk_name"] = df["risk_label"].map(risk_labels_map)

features_to_plot = ["age", "bmi", "glucose_level", "blood_pressure", "heart_rate", "sleep_hours"]
colors = ["#2ecc71", "#f39c12", "#e74c3c"]

for i, (ax, feat) in enumerate(zip(axes.flatten(), features_to_plot)):
    sns.boxplot(x="risk_name", y=feat, data=df.dropna(), order=["Low", "Medium", "High"], palette=colors, ax=ax)
    ax.set_title(feat.replace("_", " ").title(), fontsize=12)
    ax.set_xlabel("Risk Level")

plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
numeric_cols = ["age", "blood_pressure", "heart_rate", "bmi", "glucose_level", "sleep_hours", "risk_label"]

In [ ]:
corr_matrix = df[numeric_cols].corr()

In [ ]:
plt.figure(figsize=(9, 7))

In [ ]:
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn_r", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

## PART 3 : Data Pipeline: Cleaning & Preparing the Data

In [ ]:
df_engineered = df.copy()

In [ ]:
# Feature 1: Pulse Pressure
df_engineered["pulse_pressure"] = df_engineered["blood_pressure"] - 0.4 * df_engineered["heart_rate"]

In [ ]:
# Feature 2: Metabolic Risk Score
df_engineered["metabolic_risk_s"] = (
    df_engineered["bmi"].fillna(df_engineered["bmi"].median()) / 10 +
    df_engineered["glucose_level"].fillna(df_engineered["glucose_level"].median()) / 50
)

In [ ]:
activity_map   = {"High": 0, "Moderate": 1, "Low": 2}
smoking_map    = {"Never": 0, "Former": 1, "Current": 2}

In [ ]:
df_engineered["lifestyle_s"] = (
    df_engineered["activity_level"].map(activity_map) +
    df_engineered["smoking_status"].map(smoking_map) +
    (8 - df_engineered["sleep_hours"].fillna(df_engineered["sleep_hours"].median()))
)

In [ ]:
print(df_engineered[["pulse_pressure", "metabolic_risk_s", "lifestyle_s"]].describe())

In [ ]:
print("Sample rows with new features:")
df_engineered[["age", "bmi", "glucose_level", "pulse_pressure", 
               "metabolic_risk_s", "lifestyle_s"]].head(5)

In [ ]:
feature_cols = [
    "age", "gender", "blood_pressure", "heart_rate", "bmi",
    "glucose_level", "sleep_hours", "activity_level", "smoking_status",
    "pulse_pressure", "metabolic_risk_s","lifestyle_s"
]

In [ ]:
X = df_engineered[feature_cols]
y = df_engineered["risk_label"]

In [ ]:
print(f"X shape: {X.shape}  →  {X.shape[0]} patients, {X.shape[1]} features")
print(f"y shape: {y.shape}  →  {y.shape[0]} labels")
print("-----------")
print(X.head(3).to_string())
print("-----------")

##### Split intpo Training and Test sets :

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder  
from sklearn.impute import SimpleImputer              
from sklearn.pipeline import Pipeline                 
from sklearn.compose import ColumnTransformer

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= 0.2,
    random_state=42,
    stratify=y   
)

print(f"Training set : {len(X_train)} patients")
print(f"Test set : {len(X_test)} patients")

In [ ]:
print("Risk class distribution in training set:")
print(y_train.value_counts().sort_index().rename({0:"Low", 1:"Medium", 2:"High"}))
print("-----------")
print("Risk class distribution in test set:")
print(y_test.value_counts().sort_index().rename({0:"Low", 1:"Medium", 2:"High"}))
print("-----------")

##### Build the Preprocessing Pipeline

In [ ]:
numeric_cols     = ["age", "blood_pressure", "heart_rate", "bmi",
                    "glucose_level", "sleep_hours",
                    "pulse_pressure", "metabolic_risk_s", "lifestyle_s"]
categorical_cols = ["gender", "activity_level", "smoking_status"]

In [ ]:
# Pipeline for numeric columns:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

In [ ]:
# Pipeline for categorical columns:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe",     OneHotEncoder(sparse_output=False))
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer,     numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])
print("Preprocessing pipeline built!")

In [ ]:
preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)

In [ ]:
print(f"Before preprocessing - X_train shape: {X_train.shape}")
print(f"After preprocessing  - X_train shape: {X_train_processed.shape}")

## PART 4 : Machine Learning Models

##### Define all 4 models :

In [ ]:
from sklearn.linear_model import LogisticRegression    
from sklearn.tree import DecisionTreeClassifier       
from sklearn.ensemble import RandomForestClassifier, IsolationForest 
from xgboost import XGBClassifier 

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,   
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,     
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, 
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    )
}

print("4 models defined:")
for name in models:
    print(f"{name}")

##### Train and evaluate all models

In [ ]:
# Measuring how good our models are
from sklearn.metrics import (
    accuracy_score,    
    precision_score,   
    recall_score,      
    roc_auc_score,     
    classification_report  
)

In [ ]:
results = {}
trained_models = {}

In [ ]:
for model_name, clf in models.items():
    print(f"\n  Training: {model_name}...")

    full_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),  
        ("classifier",   clf)             
    ])

    full_pipeline.fit(X_train, y_train)
    y_pred  = full_pipeline.predict(X_test)
    y_proba = full_pipeline.predict_proba(X_test)

    # Calculate metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    auc  = roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted")

    # Store results
    results[model_name] = {
        "Accuracy":  round(acc,  4),
        "Precision": round(prec, 4),
        "Recall":    round(rec,  4),
        "ROC-AUC":   round(auc,  4)
    }
    trained_models[model_name] = full_pipeline

    # Print results
    print(f"  Accuracy : {acc:.1%}   Precision: {prec:.1%}")
    print(f"  Recall   : {rec:.1%}   ROC-AUC  : {auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"], digits=3))

##### Side-by-side model comparison table

In [ ]:
results_df = pd.DataFrame(results).T 
results_df = results_df.sort_values("ROC-AUC", ascending=False)

print(results_df.to_string())

best_model = results_df["ROC-AUC"].idxmax()
best_auc   = results_df["ROC-AUC"].max()
print()
print(f"Best model: {best_model}")
print(f"ROC-AUC = {best_auc:.4f}")

##### Visualize model comparison :

In [ ]:
metrics    = ["Accuracy", "Precision", "Recall", "ROC-AUC"]
model_names = list(results_df.index)
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(12, 6))

colors_m = ["#8C3929", "#8C2980", "#124229", "#663A5E"]
for i, (metric, color) in enumerate(zip(metrics, colors_m)):
    values = [results_df.loc[m, metric] for m in model_names]
    bars = ax.bar(x + i * width, values, width, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Performance Comparison", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.4)  # reference line at 1.0

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

##### Feature Importance from Random Forest:

In [ ]:
rf_pipeline = trained_models["Random Forest"]

ohe_feature_names = (rf_pipeline
                     .named_steps["preprocessor"]
                     .named_transformers_["cat"]["ohe"]
                     .get_feature_names_out(categorical_cols))

all_feature_names = numeric_cols + list(ohe_feature_names)

importances = rf_pipeline.named_steps["classifier"].feature_importances_

In [ ]:
fi_df = pd.DataFrame({
    "Feature":    all_feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False).head(12)

print("Top 12 Most Important Features:")
print(fi_df.to_string(index=False))
print()

In [ ]:
# Plot
plt.figure(figsize=(9, 6))
colors_bar = ["#e74c3c" if i < 3 else "#3498db" for i in range(len(fi_df))]
plt.barh(fi_df["Feature"][::-1], fi_df["Importance"][::-1], 
         color=colors_bar[::-1], edgecolor="white")
plt.title("Feature Importance — Random Forest", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

## PART 5 : Anomaly Detection: Finding Unusual Patients

In [ ]:
#Isolation Forest
df_anomaly = df[["age", "blood_pressure", "heart_rate", 
                      "bmi", "glucose_level", "sleep_hours"]].copy()
df_anomaly = df_anomaly.fillna(df_anomaly.median())

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.06, 
    random_state=42
)
anomaly_preds = iso_forest.fit_predict(df_anomaly)

In [ ]:
df["anomaly_iso"]  = anomaly_preds
df["is_anomaly"]   = anomaly_preds == -1

In [ ]:
n_anomalies = df["is_anomaly"].sum()
print(f"Isolation Forest Results:")
print(f"  Total patients   : {len(df)}")
print(f"  Anomalies flagged: {n_anomalies} ({n_anomalies/len(df)*100:.1f}%)")
print()
print("Sample anomalous patients:")
anomaly_sample = df[df["is_anomaly"]].head(5)[
    ["age", "blood_pressure", "heart_rate", "bmi", "glucose_level"]
]
print(anomaly_sample.to_string())

##### Domain Rule-Based Flags

In [ ]:
df["flag_tachycardia"]    = df["heart_rate"] > 110      # abnormally high HR
df["flag_hypertension"]   = df["blood_pressure"] > 160  # hypertensive crisis
df["flag_high_glucose"]   = df["glucose_level"] > 180   # diabetic threshold
df["flag_severe_obesity"]  = df["bmi"] > 38             # severe obesity
df["flag_dangerous_combo"] = (                           # dangerous combination
    (df["bmi"] > 30) &
    (df["glucose_level"] > 130) &
    (df["heart_rate"] > 95)
)

In [ ]:
print("Clinical Rule-Based Flag Summary:")
flags = {
    "Tachycardia (HR > 110 bpm)":         df["flag_tachycardia"].sum(),
    "Hypertensive Crisis (BP > 160)":     df["flag_hypertension"].sum(),
    "High Glucose (> 180 mg/dL)":         df["flag_high_glucose"].sum(),
    "Severe Obesity (BMI > 38)":          df["flag_severe_obesity"].sum(),
    "Dangerous Combination":              df["flag_dangerous_combo"].sum(),
    "Isolation Forest anomalies":         df["is_anomaly"].sum(),
}
for flag_name, count in flags.items():
    print(f"  {flag_name:40s}: {count:3d}")

##### Visualize anomalies

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Anomaly Detection Visualization", fontsize=14, fontweight="bold")

# Plot 1: Isolation Forest results
normal  = df[~df["is_anomaly"]]   # ~ means "not"
anomaly = df[df["is_anomaly"]]

axes[0].scatter(normal["heart_rate"], normal["blood_pressure"],
                c="#3498db", alpha=0.5, s=35, label="Normal", edgecolors="white", lw=0.3)
axes[0].scatter(anomaly["heart_rate"], anomaly["blood_pressure"],
                c="#e74c3c", alpha=0.9, s=80, marker="X", label="Anomaly", edgecolors="#c0392b")
axes[0].set_xlabel("Heart Rate (bpm)")
axes[0].set_ylabel("Blood Pressure (mmHg)")
axes[0].set_title("Isolation Forest: HR vs BP")
axes[0].legend()

# Plot 2: BMI vs Glucose with dangerous combo highlighted
combo_yes = df[df["flag_dangerous_combo"]]
combo_no  = df[~df["flag_dangerous_combo"]]

axes[1].scatter(combo_no["bmi"], combo_no["glucose_level"],
                c="#3498db", alpha=0.4, s=30, label="Normal combination")
axes[1].scatter(combo_yes["bmi"], combo_yes["glucose_level"],
                c="#e74c3c", alpha=0.9, s=90, marker="*", label="Dangerous combo", zorder=5)
axes[1].axvline(x=30, color="orange", linestyle="--", alpha=0.7, label="BMI=30")
axes[1].axhline(y=130, color="orange", linestyle="--", alpha=0.7, label="Glucose=130")
axes[1].set_xlabel("BMI")
axes[1].set_ylabel("Glucose Level (mg/dL)")
axes[1].set_title("Rule-Based: Dangerous Combination Flag")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("anomaly_detection.png", dpi=120, bbox_inches="tight")
plt.show()

## PART 6 : Insight Layer: Numbers → Human Language

In [ ]:
# The insight generation function
def generate_insight(row):
    risk_map   = {0: "Low", 1: "Medium", 2: "High"}
    risk_label = risk_map.get(row.get("predicted_risk", row.get("risk_label", 0)), "Unknown")

    # Find contributing factors
    reasons = []
    
    if row["bmi"] >= 30:
        reasons.append(f"elevated BMI ({row['bmi']:.1f} — obese range)")
    elif row["bmi"] >= 25:
        reasons.append(f"overweight BMI ({row['bmi']:.1f})")
    
    if row["glucose_level"] >= 126:
        reasons.append(f"high glucose ({row['glucose_level']:.0f} mg/dL, pre-diabetic/diabetic range)")
    elif row["glucose_level"] >= 100:
        reasons.append(f"borderline glucose ({row['glucose_level']:.0f} mg/dL, pre-diabetic)")
    
    if row["blood_pressure"] >= 140:
        reasons.append(f"high blood pressure ({row['blood_pressure']} mmHg, Stage 2 hypertension)")
    elif row["blood_pressure"] >= 130:
        reasons.append(f"elevated blood pressure ({row['blood_pressure']} mmHg, Stage 1 hypertension)")
    
    if row["heart_rate"] >= 100:
        reasons.append(f"elevated resting heart rate ({row['heart_rate']} bpm)")
    
    if row["sleep_hours"] < 6:
        reasons.append(f"severely insufficient sleep ({row['sleep_hours']:.1f} hrs/night, recommend ≥7)")
    elif row["sleep_hours"] < 7:
        reasons.append(f"insufficient sleep ({row['sleep_hours']:.1f} hrs/night)")
    
    if row["activity_level"] == "Low":
        reasons.append("sedentary lifestyle (low physical activity)")
    
    if row["smoking_status"] == "Current":
        reasons.append("active smoker")
    elif row["smoking_status"] == "Former":
        reasons.append("former smoker (residual risk)")
    
    if row["age"] >= 60:
        reasons.append(f"age-related elevated risk ({row['age']} years)")

    # Default if nothing triggered
    if not reasons:
        reasons = ["no major individual risk factors identified"]

    # Build recommendations

    recommendations = []

    if row["bmi"] >= 25:
        recommendations.append("structured weight management program")
    if row["glucose_level"] >= 100:
        recommendations.append("dietary sugar reduction and regular glucose monitoring")
    if row["blood_pressure"] >= 130:
        recommendations.append("blood pressure monitoring, low-sodium diet, consider medication review")
    if row["activity_level"] != "High":
        recommendations.append("increase to ≥150 min/week moderate aerobic activity (WHO guideline)")
    if row["smoking_status"] == "Current":
        recommendations.append("smoking cessation program — reduces cardiovascular risk by 50% in 1 year")
    if row["sleep_hours"] < 7:
        recommendations.append("sleep hygiene improvement (target 7–8 hrs/night)")
    
    if not recommendations:
        recommendations = ["maintain current healthy habits and schedule annual check-up"]

    #  Build the final sentence
    opening = {
        "Low":    "This patient presents with LOW overall health risk.",
        "Medium": "This patient presents with MODERATE health risk — attention advised.",
        "High":   "This patient is at HIGH health risk — prompt clinical review required."
    }[risk_label]
    
    reason_sentence = "Contributing factors: " + "; ".join(reasons) + "."
    rec_sentence    = "Recommended actions: " + "; ".join(recommendations) + "."
    
    return f"{opening}\n   {reason_sentence}\n   {rec_sentence}"

test_patient = {
    "age": 62, "bmi": 34.5, "glucose_level": 155.0,
    "blood_pressure": 152, "heart_rate": 95,
    "sleep_hours": 5.0, "activity_level": "Low",
    "smoking_status": "Current", "risk_label": 2
}

insight = generate_insight(test_patient)
print(insight)

##### Generate insights for 10 sample patients

In [ ]:
best_model_name = results_df.index[0]
best_pipeline   = trained_models[best_model_name]
print(f"Using: {best_model_name}")

In [ ]:
# Sample 10 random patients to show insights
sample_patients = df_engineered.sample(10, random_state=99).reset_index(drop=True)

for i, row in sample_patients.iterrows():
    # Make prediction for this single patient
    patient_features = sample_patients.loc[[i], feature_cols]
    predicted_risk   = best_pipeline.predict(patient_features)[0]
    predicted_proba  = best_pipeline.predict_proba(patient_features)[0]
    
    # Add prediction to row for insight generation
    row_dict = row.to_dict()
    row_dict["predicted_risk"] = predicted_risk
    
    # Generate insight
    insight = generate_insight(row_dict)
    
    conf = predicted_proba[predicted_risk]   # confidence for predicted class
    
    print(f"\nPatient #{i+1} | Confidence: {conf:.0%}")
    print(insight)

## PART 7 : Visualizations: Making Data Visual

##### Final Summary Dashboard :

In [ ]:
from sklearn.metrics import (
    accuracy_score,    
    precision_score,   
    recall_score,      
    roc_auc_score,     
    classification_report  
)

In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle("Digital Health Twin — Risk Prediction Dashboard",
             fontsize=16, fontweight="bold", y=0.98)
counts = df["risk_name"].value_counts().reindex(["Low", "Medium", "High"])
risk_colors = {"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e74c3c"}

# Panel 1: Risk Distribution
ax1 = fig.add_subplot(2, 3, 1)
bars = ax1.bar(counts.index, counts.values,
               color=[risk_colors[r] for r in counts.index],
               width=0.5, edgecolor="white")
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 2, str(val),
             ha="center", va="bottom", fontweight="bold", fontsize=11)
ax1.set_title("Risk Distribution", fontweight="bold")
ax1.set_ylabel("Patients")

# Panel 2: Feature Importance 
ax2 = fig.add_subplot(2, 3, 2)
top8 = fi_df.head(8).iloc[::-1]
ax2.barh(top8["Feature"], top8["Importance"], color="#3498db", edgecolor="white")
ax2.set_title("Top 8 Feature Importances (Random Forest)", fontweight="bold")
ax2.set_xlabel("Importance")

# Panel 3: Model Comparison (ROC-AUC)
ax3 = fig.add_subplot(2, 3, 3)
auc_vals = [results[m]["ROC-AUC"] for m in results]
model_lbls = list(results.keys())
bars3 = ax3.barh(model_lbls, auc_vals,
                  color=["#e74c3c","#e74c3c","#27ae60","#2ecc71"],
                  edgecolor="white")
for bar, val in zip(bars3, auc_vals):
    ax3.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.4f}", ha="right", va="center", color="white", fontweight="bold")
ax3.set_title("ROC-AUC Comparison", fontweight="bold")
ax3.set_xlabel("ROC-AUC Score")
ax3.set_xlim(0.85, 1.02)

# Panel 4: BMI vs Glucose 
ax4 = fig.add_subplot(2, 3, 4)
for risk_name, group in df.groupby("risk_name"):
    ax4.scatter(group["bmi"], group["glucose_level"],
                c=risk_colors[risk_name], label=risk_name, alpha=0.5, s=25)
ax4.set_title("BMI vs Glucose by Risk", fontweight="bold")
ax4.set_xlabel("BMI"); ax4.set_ylabel("Glucose (mg/dL)")
ax4.legend(fontsize=9)

# Panel 5: Anomaly Detection 
ax5 = fig.add_subplot(2, 3, 5)
ax5.scatter(df[~df["is_anomaly"]]["heart_rate"],
            df[~df["is_anomaly"]]["blood_pressure"],
            c="#3498db", alpha=0.4, s=25, label="Normal")
ax5.scatter(df[df["is_anomaly"]]["heart_rate"],
            df[df["is_anomaly"]]["blood_pressure"],
            c="#e74c3c", alpha=0.9, s=60, marker="X", label="Anomaly")
ax5.set_title("Anomaly: HR vs BP", fontweight="bold")
ax5.set_xlabel("Heart Rate (bpm)"); ax5.set_ylabel("BP (mmHg)")
ax5.legend(fontsize=9)

# Panel 6: Correlation Heatmap 
ax6 = fig.add_subplot(2, 3, 6)
corr_small = df[["age","bmi","glucose_level","blood_pressure","heart_rate","risk_label"]].corr()
sns.heatmap(corr_small, annot=True, fmt=".2f", cmap="RdYlGn_r",
            ax=ax6, linewidths=0.5, cbar=False, vmin=-1, vmax=1)
ax6.set_title("Correlation Matrix", fontweight="bold")

plt.tight_layout()
plt.savefig("dashboard.png", dpi=120, bbox_inches="tight")
plt.show()